# 节点 3：至 run_DSSC.py:170，脚本执行完毕——fit 返回、最终评估打印完、latent.csv_i 与 pred.csv_i 写入 src/out_151507/

In [18]:
# 获取先前保存的变量
import numpy as np
import scipy.sparse as sp
import torch
import sys, pathlib
sys.path.append(str(pathlib.Path("../../src").resolve()))
from DSSC import DSSC

ckpt = torch.load("data/model_final.pt", map_location="cpu")
model = DSSC(**ckpt["config"], device="cuda").to("cuda")
sd = {k: v for k, v in ckpt["state_dict"].items() if not k.startswith("mu")}
model.load_state_dict(sd, strict=False)

<All keys matched successfully>

In [19]:
y_pred = np.load("./data/y_pred.npy")

fit_result = np.load("./data/fit_result.npz")
final_loss = fit_result["final_loss"]
epoch = fit_result["epoch"]

In [20]:
d = np.load("./data/arrays.npz", allow_pickle=True)
p_ = d["p_"]
p_ = [np.asarray(x, dtype=int) for x in d["p_"]]
y = d["y"]

In [31]:
A0 = sp.load_npz("data/A0.npz").toarray()
A_n = sp.load_npz("data/A_n.npz")

In [27]:
import scanpy as sc

adata = sc.read_h5ad("./data/adata.h5ad")

## 1. fit 返回（DSSC.py:340 → run_DSSC.py:156）
return self.y_pred, total_loss, epoch+1：
- y_pred：(4221,) 最终硬聚类标签（0–6）；
- total_loss：最后一个 epoch 的 loss（如日志 Loss = 0.00011715）；
- epoch+1：实际跑的轮数（早停触发时的轮数，如 run1 的 34）。
早停由 DSSC.py:279-282 触发：delta_label < tol=0.001。注意早停是在更新之前 break 的，所以最后一次参数更新属于上一轮。

## 2. 最终评估（run_DSSC.py:159-163）
此刻数据仍是隐空间 (4221,32) 与标签 (4221,)：
1. eval_cluster(y, y_pred)（utils.py:57-62 → best_map utils.py:25-47）：
用 Munkres（匈牙利算法）做标签最优匹配后算 ACC，另算 NMI、ARI。
2. Iscore_label(y_pred+1., A0)（utils.py:77-88）：A0 是仅用于评估的 k=20 空间 kNN 图（include_self=False，run_DSSC.py:110-112）；该指标衡量标签在空间邻域上的一致性（y_pred+1 避免 0 的倒数）。
3. knn_ACC(p_, y_pred)（utils.py:90-97）：p_ 是每个细胞的 20 个最近邻索引（run_DSSC.py:114-118）；统计某细胞标签与其空间邻居多数标签一致的比例。
4. 打印 Final Clustering: ACC=... NMI=... ARI=... kNN_ACC=... I_score=...。

In [22]:
from utils import *

acc, nmi, ari = eval_cluster(y, y_pred)
Iscore = Iscore_label(y_pred+1., A0)
ka = knn_ACC(p_, y_pred)
print('Final Clustering: ACC= %.4f, NMI= %.4f, ARI= %.4f, kNN_ACC= %.4f, I_score= %.4f, Loss = %.8f, Epoch = %.i' % (acc, nmi, ari, ka, Iscore, final_loss, epoch))

/hdd2/liuzy/Projects/IntensiveStudy4Lin/DSSC/src/utils.py:94: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to True or False to avoid this warning.
  l_mode = stats.mode(labels).mode[0]


Final Clustering: ACC= 0.6382, NMI= 0.6424, ARI= 0.5229, kNN_ACC= 0.9621, I_score= 0.8308, Loss = 0.00006880, Epoch = 348


20

## 3. 落盘（run_DSSC.py:165-170）
- 隐表示：final_latent = model.encodeBatch(adata.X, A_n) → (4221,32)，np.savetxt(out_151507/latent.csv_{run})（逗号分隔，32 列）。
- 预测标签：np.savetxt(out_151507/pred.csv_{run}, y_pred) → (4221,1)，浮点格式（如 4.0、0.0）。
- for i in {1..5} 会产生 latent.csv_1..5 与 pred.csv_1..5，各 4221 行。

In [32]:
# if args.final_latent_file != -1:
final_latent = model.encodeBatch(torch.tensor(adata.X, dtype=torch.float32), A_n).data.cpu().numpy()
np.savetxt("../../src/out_151507" + "/" + "latent.csv" + "_" + "pseudodebug", final_latent, delimiter=",")   
# if args.final_labels != -1:
np.savetxt("../../src/out_151507" + "/" + "pred.csv" + "_" + "pseudodebug", y_pred, delimiter=",")

# 全流程总结（输入 → 模块 → 输出）
| 输入 | 关键变换 | 进入的模块 | 最终去向 |
| ---- | ------ | -------- | ------- |
| h5 X (4226×24402) | 剔 NA → HVG 2000 → log 归一化 / 原始 counts 两份 | GAT 编码器输入 + ZINB 重建目标 | 隐表示 latent.csv_i、聚类标签 pred.csv_i |
| h5 Pos (2×4226) | 转置 → kNN=20 → 归一化邻接 | GAT 图 A_n（预训练+聚类）；A0/p_ 仅评估 | 影响隐表示与评估指标 |
| h5 Y (4226,) | 剔 NA → cat.codes | 仅用于 n_clusters 与每轮/最终评估 | 不进梯度 |
| *_mlFromMarks_test.txt | 读入−1 → 每轮抽 4000 对 | pairwise_loss(...,"ML")（偶数轮） | 塑造隐空间，间接影响输出 |
| *_clFromMarks_test.txt | 读入−1 → 每轮抽 4000 对 | pairwise_loss(...,"CL")（奇数轮） | 同上 |